In [1]:
import pandas as pd
import numpy as np

# Dataset "sucio" de tickets de farmacia
datos = {
    "id_ticket": [1001, 1002, 1003, 1004, 1005, 1003, 1006, 1007, 1008, 1009],
    "fecha": ["2026-05-01", "2026-05-01", "2026-05-02", "2026-05-02", "2026-05-03", 
              "2026-05-02", "2026-05-03", "05/04/2026", "2026-05-04", "2026-05-05"],
    "Articulo": ["paracetamol", "Café Americano", "ibuprofeno", None, "agua", 
                 "ibuprofeno", "VITAMINA C", "café americano", "Agua", np.nan],
    "categoria": ["medicamento", "café", "medicamento", "bebida", "bebida", 
                   "medicamento", "medicamento", None, "Bebida", "café"],
    "Cantidad": [2, 1, 1, 3, "dos", 1, 1, 2, -1, 1],
    "precio_unitario": [25.50, 18.0, 35.0, 15.0, 15.0, 35.0, 42.0, 18.0, 15.0, None],
}

df = pd.DataFrame(datos)
df

,id_ticket,fecha,Articulo,categoria,Cantidad,precio_unitario
0,1001,2026-05-01,paracetamol,medicamento,2,25.5
1,1002,2026-05-01,Café Americano,café,1,18.0
2,1003,2026-05-02,ibuprofeno,medicamento,1,35.0
3,1004,2026-05-02,NaN,bebida,3,15.0
4,1005,2026-05-03,agua,bebida,dos,15.0
5,1003,2026-05-02,ibuprofeno,medicamento,1,35.0
6,1006,2026-05-03,VITAMINA C,medicamento,1,42.0
7,1007,05/04/2026,café americano,NaN,2,18.0
8,1008,2026-05-04,Agua,Bebida,-1,15.0
9,1009,2026-05-05,NaN,café,1,NaN


In [2]:
# PASO 1: Estandarizar nombres de columnas
df.columns = df.columns.str.lower()
print(df.columns.tolist())

['id_ticket', 'fecha', 'articulo', 'categoria', 'cantidad', 'precio_unitario']


In [3]:
# PASO 2: Estandarizar texto a minúsculas
df["articulo"] = df["articulo"].str.lower()
df["categoria"] = df["categoria"].str.lower()
df[["articulo", "categoria"]]

,articulo,categoria
0,paracetamol,medicamento
1,café americano,café
2,ibuprofeno,medicamento
3,NaN,bebida
4,agua,bebida
5,ibuprofeno,medicamento
6,vitamina c,medicamento
7,café americano,NaN
8,agua,bebida
9,NaN,café


In [4]:
# PASO 3: Diagnóstico de nulos
print(df.isnull().sum())
print("---")
print(f"Total filas: {len(df)}")

id_ticket          0
fecha              0
articulo           2
categoria          1
cantidad           0
precio_unitario    1
dtype: int64
---
Total filas: 10


In [5]:
# Ver las filas con al menos un nulo
df[df.isnull().any(axis=1)]

,id_ticket,fecha,articulo,categoria,cantidad,precio_unitario
3,1004,2026-05-02,NaN,bebida,3,15.0
7,1007,05/04/2026,café americano,NaN,2,18.0
9,1009,2026-05-05,NaN,café,1,NaN


In [6]:
# PASO 4: Eliminar filas irrecuperables
df = df.drop(df[(df["id_ticket"] == 1004) | (df["id_ticket"] == 1009)].index)

# Rellenar lo que sí podemos rescatar
df.loc[df["id_ticket"] == 1007, "categoria"] = "café"

print(df.isnull().sum())
print(f"Filas restantes: {len(df)}")

id_ticket          0
fecha              0
articulo           0
categoria          0
cantidad           0
precio_unitario    0
dtype: int64
Filas restantes: 8


In [7]:
# PASO 5: Detectar duplicados
print(df[df.duplicated(subset="id_ticket", keep=False)])

   id_ticket       fecha    articulo    categoria cantidad  precio_unitario
2       1003  2026-05-02  ibuprofeno  medicamento        1             35.0
5       1003  2026-05-02  ibuprofeno  medicamento        1             35.0


In [8]:
# Eliminar duplicados, quedarnos con la primera aparición
df = df.drop_duplicates(subset="id_ticket", keep="first")
print(f"Filas restantes: {len(df)}")

Filas restantes: 7


In [9]:
# PASO 6: Revisar tipos de datos
print(df.dtypes)

id_ticket            int64
fecha                  str
articulo               str
categoria              str
cantidad            object
precio_unitario    float64
dtype: object


In [10]:
# PASO 7: Convertir cantidad a numérico
df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce")
print(df[["id_ticket", "cantidad"]])

   id_ticket  cantidad
0       1001       2.0
1       1002       1.0
2       1003       1.0
4       1005       NaN
6       1006       1.0
7       1007       2.0
8       1008      -1.0


In [11]:
# Eliminar fila con cantidad nula
df = df.dropna(subset=["cantidad"])

# Corregir valor negativo
df.loc[df["cantidad"] < 0, "cantidad"] = df.loc[df["cantidad"] < 0, "cantidad"].abs()

print(df[["id_ticket", "cantidad"]])

   id_ticket  cantidad
0       1001       2.0
1       1002       1.0
2       1003       1.0
6       1006       1.0
7       1007       2.0
8       1008       1.0


In [12]:
# PASO 8: Estandarizar fechas
print(df["fecha"].values)
df["fecha"] = pd.to_datetime(df["fecha"], format="mixed")
print(df.dtypes)

<ArrowStringArray>
['2026-05-01', '2026-05-01', '2026-05-02', '2026-05-03', '05/04/2026',
 '2026-05-04']
Length: 6, dtype: str
id_ticket                   int64
fecha              datetime64[us]
articulo                      str
categoria                     str
cantidad                  float64
precio_unitario           float64
dtype: object


In [13]:
# RESULTADO FINAL
print(f"Filas: {len(df)}")
print(f"Nulos: {df.isnull().sum().sum()}")
print(f"Duplicados: {df.duplicated(subset='id_ticket').sum()}")
print("---")
df

Filas: 6
Nulos: 0
Duplicados: 0
---


,id_ticket,fecha,articulo,categoria,cantidad,precio_unitario
0,1001,2026-05-01,paracetamol,medicamento,2.0,25.5
1,1002,2026-05-01,café americano,café,1.0,18.0
2,1003,2026-05-02,ibuprofeno,medicamento,1.0,35.0
6,1006,2026-05-03,vitamina c,medicamento,1.0,42.0
7,1007,2026-05-04,café americano,café,2.0,18.0
8,1008,2026-05-04,agua,bebida,1.0,15.0


In [14]:
def limpiar_tickets(df):
    """Pipeline de limpieza para tickets de farmacia."""
    # 1. Estandarizar nombres de columnas
    df.columns = df.columns.str.lower()
    
    # 2. Estandarizar texto a minúsculas
    df["articulo"] = df["articulo"].str.lower()
    df["categoria"] = df["categoria"].str.lower()
    
    # 3. Eliminar duplicados
    df = df.drop_duplicates(subset="id_ticket", keep="first")
    
    # 4. Convertir tipos de datos
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce")
    
    # 5. Eliminar nulos críticos
    df = df.dropna(subset=["articulo", "cantidad", "precio_unitario"])
    
    # 6. Estandarizar fechas
    df["fecha"] = pd.to_datetime(df["fecha"], format="mixed")
    
    return df